# QueleaGuard — Ngao Presentation Model Training

## Purpose

This notebook develops the temporary machine-learning models used for the Ngao Labs presentation.

The models are trained from the validated Ngao Presentation dataset and are intended for **presentation/demo purposes**, not as the final production QueleaGuard prediction system.

### Models

- Logistic Regression
- Random Forest

### Validation principle

Repeated environmental feature profiles are kept together during train/test splitting to prevent identical profiles from appearing in both partitions.

### Dataset

`data/processed/ngao_presentation_dataset.csv`

### Output

Trained presentation model artifacts will be saved under:

`models/ngao_presentation/`

> **Important:** These models must not be represented as the final production QueleaGuard models. The production modelling pipeline remains a separate engineering phase.

In [12]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\pc\Projects\QueleaGuard\venv\Scripts\python.exe
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [13]:
import sys
import os
import importlib.util
from pathlib import Path

print("=" * 60)
print("QUELEAGUARD — ENVIRONMENT STATUS")
print("=" * 60)

# Python
print("\n[1] PYTHON")
print("Version    :", sys.version.split()[0])
print("Executable :", sys.executable)

# Working directory
print("\n[2] WORKING DIRECTORY")
print("Current directory:", os.getcwd())
print("Project root exists:", Path("data").exists() and Path("src").exists())

# Packages
print("\n[3] REQUIRED PACKAGES")

packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scikit-learn": "sklearn",
    "ipykernel": "ipykernel",
    "jupyter": "jupyter",
}

for display_name, import_name in packages.items():
    try:
        module = __import__(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"✓ {display_name:<15} {version}")
    except ImportError:
        print(f"✗ {display_name:<15} NOT AVAILABLE")

# Dataset
print("\n[4] QUELEAGUARD DATASET")

dataset = Path("data/processed/ngao_presentation_dataset.csv")

if dataset.exists():
    print("✓ Dataset found")
    print("  Path:", dataset.resolve())
else:
    print("✗ Dataset NOT found from current directory")

# Notebook
print("\n[5] NOTEBOOK ENVIRONMENT")
print("Python prefix:", sys.prefix)
print("Virtual environment:",
      "YES" if "venv" in sys.prefix.lower() else "CHECK")

print("\n" + "=" * 60)
print("STATUS CHECK COMPLETE")
print("=" * 60)

QUELEAGUARD — ENVIRONMENT STATUS

[1] PYTHON
Version    : 3.14.3
Executable : c:\Users\pc\Projects\QueleaGuard\venv\Scripts\python.exe

[2] WORKING DIRECTORY
Current directory: c:\Users\pc\Projects\QueleaGuard\notebooks
Project root exists: False

[3] REQUIRED PACKAGES
✓ pandas          3.0.5
✓ numpy           2.5.1
✓ scikit-learn    1.9.0
✓ ipykernel       7.3.0
✓ jupyter         unknown

[4] QUELEAGUARD DATASET
✗ Dataset NOT found from current directory

[5] NOTEBOOK ENVIRONMENT
Python prefix: c:\Users\pc\Projects\QueleaGuard\venv
Virtual environment: YES

STATUS CHECK COMPLETE


## 2. Project Paths and Dataset Configuration

The notebook is executed from the `notebooks/` directory.  
Project-level resources are therefore referenced relative to the notebook location.

The Ngao Presentation dataset is stored under:

`data/processed/ngao_presentation_dataset.csv`

Model artifacts will be saved under:

`models/ngao_presentation/`

In [14]:
from pathlib import Path

# Notebook directory
NOTEBOOK_DIR = Path.cwd()

# QueleaGuard project root
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Project paths
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ngao_presentation_dataset.csv"
MODEL_DIR = PROJECT_ROOT / "models" / "ngao_presentation"

# Create model output directory if it doesn't exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook directory:")
print(NOTEBOOK_DIR)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nDataset:")
print(DATA_PATH)

print("\nDataset exists:", DATA_PATH.exists())

print("\nModel output directory:")
print(MODEL_DIR)

Notebook directory:
c:\Users\pc\Projects\QueleaGuard\notebooks

Project root:
c:\Users\pc\Projects\QueleaGuard

Dataset:
c:\Users\pc\Projects\QueleaGuard\data\processed\ngao_presentation_dataset.csv

Dataset exists: True

Model output directory:
c:\Users\pc\Projects\QueleaGuard\models\ngao_presentation


## 3. Load the Ngao Presentation Dataset

Load the validated Ngao Presentation dataset from the processed-data directory and inspect its structure before modelling.

In [15]:
import pandas as pd

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully.
Shape: (259, 15)

Columns:
['rainfall_7d', 'rainfall_30d', 'rainfall_90d', 'temp_mean_7d', 'dewpoint_mean_7d', 'wind_mean_7d', 'temp_same_day', 'dewpoint_same_day', 'wind_same_day', 'ndvi_nearest_composite', 'ndvi_anomaly', 'elevation_m', 'slope_deg', 'dist_to_water_m', 'presence']

First 5 rows:


,rainfall_7d,rainfall_30d,rainfall_90d,temp_mean_7d,dewpoint_mean_7d,wind_mean_7d,temp_same_day,dewpoint_same_day,wind_same_day,ndvi_nearest_composite,ndvi_anomaly,elevation_m,slope_deg,dist_to_water_m,presence
0,70.49,310.08,573.74,27.81,15.28,1.86,24.28,16.06,0.87,0.6869,0.1463,1158.0,0.98,231.0,1
1,70.49,310.08,573.74,27.81,15.28,1.86,24.28,16.06,0.87,0.6869,0.1463,1158.0,0.98,231.0,1
2,57.97,149.83,613.94,24.46,18.41,3.18,23.84,19.16,3.76,0.6300,0.0614,1134.0,2.18,131.6,1
3,57.97,149.83,613.94,24.46,18.41,3.18,23.84,19.16,3.76,0.6300,0.0614,1134.0,2.18,131.6,1
4,45.27,118.18,470.30,30.25,14.85,2.26,29.34,15.75,2.24,0.5334,0.0111,1158.0,0.98,231.0,1


## 4. Dataset Validation

Before modelling, verify the dataset dimensions, target distribution, missing values, and feature structure.

In [16]:
# Dataset dimensions
print("Dataset shape:", df.shape)

# Column structure
print("\nColumns:")
print(df.columns.tolist())

# Target distribution
print("\nTarget value counts:")
print(df["presence"].value_counts())

print("\nTarget proportions:")
print(df["presence"].value_counts(normalize=True))

# Missing values
print("\nMissing values by column:")
print(df.isna().sum())

print("\nTotal missing values:", df.isna().sum().sum())

Dataset shape: (259, 15)

Columns:
['rainfall_7d', 'rainfall_30d', 'rainfall_90d', 'temp_mean_7d', 'dewpoint_mean_7d', 'wind_mean_7d', 'temp_same_day', 'dewpoint_same_day', 'wind_same_day', 'ndvi_nearest_composite', 'ndvi_anomaly', 'elevation_m', 'slope_deg', 'dist_to_water_m', 'presence']

Target value counts:
presence
1    133
0    126
Name: count, dtype: int64

Target proportions:
presence
1    0.513514
0    0.486486
Name: proportion, dtype: float64

Missing values by column:
rainfall_7d               0
rainfall_30d              0
rainfall_90d              0
temp_mean_7d              0
dewpoint_mean_7d          0
wind_mean_7d              0
temp_same_day             0
dewpoint_same_day         0
wind_same_day             0
ndvi_nearest_composite    0
ndvi_anomaly              0
elevation_m               0
slope_deg                 0
dist_to_water_m           0
presence                  0
dtype: int64

Total missing values: 0


## 5. Define Features and Target

Separate the environmental predictors from the target variable.

- `X` contains the 14 environmental features.
- `y` contains the binary `presence` target.

In [17]:
# Separate features and target

X = df.drop(columns=["presence"])
y = df["presence"]

print("Number of features:", X.shape[1])
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget dtype:", y.dtype)

Number of features: 14
X shape: (259, 14)
y shape: (259,)

Feature columns:
['rainfall_7d', 'rainfall_30d', 'rainfall_90d', 'temp_mean_7d', 'dewpoint_mean_7d', 'wind_mean_7d', 'temp_same_day', 'dewpoint_same_day', 'wind_same_day', 'ndvi_nearest_composite', 'ndvi_anomaly', 'elevation_m', 'slope_deg', 'dist_to_water_m']

Target dtype: int64


## 6. Identify Repeated Environmental Profiles

Multiple observations may share exactly the same environmental feature values.

If identical environmental profiles are split across training and testing, the model could effectively encounter the same conditions in both partitions.

We therefore identify repeated profiles and use them as validation groups.

In [18]:
# Identify repeated environmental profiles

feature_columns = X.columns.tolist()

profile_counts = (
    df.groupby(feature_columns, dropna=False)
      .size()
      .sort_values(ascending=False)
)

total_rows = len(df)
unique_profiles = len(profile_counts)

repeated_profiles = profile_counts[profile_counts > 1]

rows_in_repeated_profiles = repeated_profiles.sum()

print("Total rows:", total_rows)
print("Unique environmental profiles:", unique_profiles)
print("Profiles occurring more than once:", len(repeated_profiles))
print("Rows belonging to repeated profiles:", rows_in_repeated_profiles)

print("\nMost frequently repeated profiles:")
print(repeated_profiles.head(10))


Total rows: 259
Unique environmental profiles: 155
Profiles occurring more than once: 60
Rows belonging to repeated profiles: 164

Most frequently repeated profiles:
rainfall_7d  rainfall_30d  rainfall_90d  temp_mean_7d  dewpoint_mean_7d  wind_mean_7d  temp_same_day  dewpoint_same_day  wind_same_day  ndvi_nearest_composite  ndvi_anomaly  elevation_m  slope_deg  dist_to_water_m
0.00         40.92         385.37        26.17         13.26             2.07          28.21          11.19              0.44            0.6030                 -0.0166       1448.0       0.69       1174.5             5
21.68        84.05         636.37        26.06         14.89             1.76          25.99          15.18              2.15            0.0045                 -0.5446       1134.0       2.18       131.6              5
             77.39         625.56        25.69         15.31             1.82          25.31          15.89              1.95            0.0045                 -0.5446       1134.0  

## 7. Check Label Consistency Within Repeated Profiles

Before using environmental profiles as validation groups, verify that repeated profiles have a consistent target label.

A repeated profile with both `presence = 1` and `presence = 0` would indicate conflicting labels for identical environmental conditions.

In [19]:
# Check whether repeated environmental profiles have conflicting labels

label_counts = (
    df.groupby(feature_columns)["presence"]
      .nunique()
)

conflicting_profiles = label_counts[label_counts > 1]

print("Repeated profiles with conflicting labels:", len(conflicting_profiles))

if len(conflicting_profiles) == 0:
    print("All repeated environmental profiles have a consistent presence label.")
else:
    print("\nConflicting profiles:")
    print(conflicting_profiles)

Repeated profiles with conflicting labels: 0
All repeated environmental profiles have a consistent presence label.


## 8. Create Environmental-Profile Validation Groups

Each unique combination of the 14 environmental features is assigned a single group identifier.

All rows sharing the same environmental profile will therefore remain in the same validation group.

This prevents identical environmental profiles from being distributed across different validation partitions.

In [20]:
# Create a validation group for each unique environmental profile

groups = (
    df[feature_columns]
    .astype(str)
    .agg("|".join, axis=1)
)

print("Total rows:", len(df))
print("Unique validation groups:", groups.nunique())
print("Groups with multiple rows:", groups.value_counts().gt(1).sum())

Total rows: 259
Unique validation groups: 155
Groups with multiple rows: 60


## 9. Create a Grouped and Stratified Validation Split

Use `StratifiedGroupKFold` so that:

1. The presence/pseudo-absence distribution remains reasonably balanced.
2. Identical environmental profiles never appear in both training and validation data.
3. Each group is kept entirely within one partition.

In [21]:
from sklearn.model_selection import StratifiedGroupKFold

# Create a 5-fold grouped and stratified cross-validation splitter
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Inspect the folds
for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    print(f"\nFold {fold}")
    print("-" * 30)

    print("Training rows:", len(train_idx))
    print("Validation rows:", len(test_idx))

    print(
        "Training groups:",
        groups.iloc[train_idx].nunique()
    )

    print(
        "Validation groups:",
        groups.iloc[test_idx].nunique()
    )

    print(
        "Training presence rate:",
        round(y.iloc[train_idx].mean(), 3)
    )

    print(
        "Validation presence rate:",
        round(y.iloc[test_idx].mean(), 3)
    )


Fold 1
------------------------------
Training rows: 207
Validation rows: 52
Training groups: 124
Validation groups: 31
Training presence rate: 0.512
Validation presence rate: 0.519

Fold 2
------------------------------
Training rows: 207
Validation rows: 52
Training groups: 124
Validation groups: 31
Training presence rate: 0.517
Validation presence rate: 0.5

Fold 3
------------------------------
Training rows: 207
Validation rows: 52
Training groups: 124
Validation groups: 31
Training presence rate: 0.512
Validation presence rate: 0.519

Fold 4
------------------------------
Training rows: 207
Validation rows: 52
Training groups: 123
Validation groups: 32
Training presence rate: 0.512
Validation presence rate: 0.519

Fold 5
------------------------------
Training rows: 208
Validation rows: 51
Training groups: 125
Validation groups: 30
Training presence rate: 0.514
Validation presence rate: 0.51


## 10. Verify Group Separation

Confirm that no environmental validation group appears in both the training and validation partitions of any fold.

A correctly grouped split must have zero overlapping groups.

In [22]:
# Verify that no validation group appears in both partitions

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    train_groups = set(groups.iloc[train_idx])
    val_groups = set(groups.iloc[val_idx])

    overlap = train_groups.intersection(val_groups)

    print(f"Fold {fold}:")
    print(f"  Training groups:   {len(train_groups)}")
    print(f"  Validation groups: {len(val_groups)}")
    print(f"  Overlapping groups: {len(overlap)}")

    if len(overlap) == 0:
        print("  ✓ No group leakage")
    else:
        print("  ✗ GROUP LEAKAGE DETECTED")

    print()

Fold 1:
  Training groups:   124
  Validation groups: 31
  Overlapping groups: 0
  ✓ No group leakage

Fold 2:
  Training groups:   124
  Validation groups: 31
  Overlapping groups: 0
  ✓ No group leakage

Fold 3:
  Training groups:   124
  Validation groups: 31
  Overlapping groups: 0
  ✓ No group leakage

Fold 4:
  Training groups:   123
  Validation groups: 32
  Overlapping groups: 0
  ✓ No group leakage

Fold 5:
  Training groups:   125
  Validation groups: 30
  Overlapping groups: 0
  ✓ No group leakage



## 11. Establish a Majority-Class Baseline

Before evaluating machine-learning models, establish a simple baseline.

The majority-class baseline always predicts the most common target class in the training data.

A useful model should demonstrate performance beyond this baseline under the same grouped validation strategy.

In [23]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Store fold-level baseline results
baseline_results = []

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Determine majority class from training data only
    majority_class = y_train.mode()[0]

    # Predict majority class for validation data
    y_pred = [majority_class] * len(y_val)

    accuracy = accuracy_score(y_val, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_val, y_pred)

    baseline_results.append({
        "fold": fold,
        "majority_class": majority_class,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy
    })

baseline_results_df = pd.DataFrame(baseline_results)

display(baseline_results_df)

print(
    "\nMean accuracy:",
    round(baseline_results_df["accuracy"].mean(), 3)
)

print(
    "Mean balanced accuracy:",
    round(baseline_results_df["balanced_accuracy"].mean(), 3)
)

,fold,majority_class,accuracy,balanced_accuracy
0,1,1,0.519231,0.5
1,2,1,0.500000,0.5
2,3,1,0.519231,0.5
3,4,1,0.519231,0.5
4,5,1,0.509804,0.5



Mean accuracy: 0.513
Mean balanced accuracy: 0.5


## 12. Logistic Regression

Train a Logistic Regression model using the same grouped and stratified five-fold validation strategy.

Feature standardization is performed inside a pipeline so that scaling parameters are learned exclusively from each training fold.

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

logistic_results = []

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Build model pipeline
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    # Train only on the training fold
    model.fit(X_train, y_train)

    # Predict validation fold
    y_pred = model.predict(X_val)

    # Calculate metrics
    accuracy = accuracy_score(y_val, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    logistic_results.append({
        "fold": fold,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

logistic_results_df = pd.DataFrame(logistic_results)

display(logistic_results_df)

print("\nMean metrics:")
print(
    logistic_results_df[
        ["accuracy", "balanced_accuracy", "precision", "recall", "f1"]
    ].mean().round(3)
)

,fold,accuracy,balanced_accuracy,precision,recall,f1
0,1,0.884615,0.881481,0.838710,0.962963,0.896552
1,2,0.750000,0.750000,0.782609,0.692308,0.734694
2,3,0.788462,0.781481,0.722222,0.962963,0.825397
3,4,0.923077,0.921481,0.896552,0.962963,0.928571
4,5,0.921569,0.923077,1.000000,0.846154,0.916667



Mean metrics:
accuracy             0.854
balanced_accuracy    0.852
precision            0.848
recall               0.885
f1                   0.860
dtype: float64


## 13. Random Forest

Train a Random Forest classifier using the same grouped and stratified five-fold validation strategy.

The Random Forest provides a nonlinear comparison against Logistic Regression and can capture interactions between environmental variables.

In [25]:
from sklearn.ensemble import RandomForestClassifier

random_forest_results = []

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # Build Random Forest model
    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    # Train only on the training fold
    model.fit(X_train, y_train)

    # Predict validation fold
    y_pred = model.predict(X_val)

    # Calculate metrics
    accuracy = accuracy_score(y_val, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    random_forest_results.append({
        "fold": fold,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

random_forest_results_df = pd.DataFrame(random_forest_results)

display(random_forest_results_df)

print("\nMean metrics:")
print(
    random_forest_results_df[
        ["accuracy", "balanced_accuracy", "precision", "recall", "f1"]
    ].mean().round(3)
)

,fold,accuracy,balanced_accuracy,precision,recall,f1
0,1,0.942308,0.940000,0.900000,1.000000,0.947368
1,2,0.750000,0.750000,0.782609,0.692308,0.734694
2,3,0.730769,0.725926,0.696970,0.851852,0.766667
3,4,0.903846,0.902963,0.892857,0.925926,0.909091
4,5,0.960784,0.961538,1.000000,0.923077,0.960000



Mean metrics:
accuracy             0.858
balanced_accuracy    0.856
precision            0.854
recall               0.879
f1                   0.864
dtype: float64


## 14. Compare Model Performance

Compare the majority baseline, Logistic Regression, and Random Forest using the same grouped validation framework.

In [26]:
# Summarize model performance

model_comparison = pd.DataFrame({
    "Model": [
        "Majority Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        baseline_results_df["accuracy"].mean(),
        logistic_results_df["accuracy"].mean(),
        random_forest_results_df["accuracy"].mean()
    ],
    "Balanced Accuracy": [
        baseline_results_df["balanced_accuracy"].mean(),
        logistic_results_df["balanced_accuracy"].mean(),
        random_forest_results_df["balanced_accuracy"].mean()
    ],
    "Precision": [
        float("nan"),
        logistic_results_df["precision"].mean(),
        random_forest_results_df["precision"].mean()
    ],
    "Recall": [
        float("nan"),
        logistic_results_df["recall"].mean(),
        random_forest_results_df["recall"].mean()
    ],
    "F1": [
        float("nan"),
        logistic_results_df["f1"].mean(),
        random_forest_results_df["f1"].mean()
    ]
})

display(model_comparison.round(3))

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1
0,Majority Baseline,0.513,0.500,NaN,NaN,NaN
1,Logistic Regression,0.854,0.852,0.848,0.885,0.860
2,Random Forest,0.858,0.856,0.854,0.879,0.864


## 15. Fit Presentation Models for Interpretation

Fit Logistic Regression and Random Forest on the complete Ngao Presentation dataset for model interpretation.

These full-data models are used to inspect feature relationships and importance after validation. They are not used to estimate validation performance.

In [27]:
# Fit Logistic Regression on the complete dataset

logistic_model_full = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logistic_model_full.fit(X, y)

# Fit Random Forest on the complete dataset

random_forest_model_full = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

random_forest_model_full.fit(X, y)

print("Both presentation models trained successfully.")

Both presentation models trained successfully.


## 16. Random Forest Feature Importance

Inspect the relative contribution of each environmental feature to the Random Forest's predictions.

These importance scores indicate how useful each feature was to the fitted model. They do not establish causation.

In [28]:
# Extract Random Forest feature importance

rf_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": random_forest_model_full.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

display(rf_importance)

,feature,importance
0,elevation_m,0.235308
1,slope_deg,0.149401
2,dist_to_water_m,0.132561
3,ndvi_nearest_composite,0.093289
4,dewpoint_mean_7d,0.060300
5,ndvi_anomaly,0.060017
6,rainfall_7d,0.044368
7,dewpoint_same_day,0.039534
8,temp_mean_7d,0.039312
9,rainfall_30d,0.037767


## 17. Logistic Regression Coefficients

Inspect the standardized Logistic Regression coefficients.

Positive coefficients indicate that higher standardized values of a feature are associated with a higher predicted probability of the `presence` class, while negative coefficients indicate the opposite.

These associations are model-specific and should not be interpreted as causal relationships.

In [29]:
# Extract Logistic Regression coefficients

logistic_coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": logistic_model_full.named_steps["classifier"].coef_[0]
})

logistic_coefficients["absolute_coefficient"] = (
    logistic_coefficients["coefficient"].abs()
)

logistic_coefficients = (
    logistic_coefficients
    .sort_values("absolute_coefficient", ascending=False)
    .reset_index(drop=True)
)

display(logistic_coefficients)

,feature,coefficient,absolute_coefficient
0,elevation_m,-3.543484,3.543484
1,wind_mean_7d,-1.080179,1.080179
2,dist_to_water_m,-0.891004,0.891004
3,rainfall_7d,0.771147,0.771147
4,temp_same_day,0.688120,0.688120
5,rainfall_30d,0.619811,0.619811
6,slope_deg,-0.586274,0.586274
7,dewpoint_mean_7d,0.541198,0.541198
8,rainfall_90d,-0.540631,0.540631
9,dewpoint_same_day,0.489213,0.489213


## 18. Examine Feature Correlations

Inspect correlations among environmental predictors to identify strongly related variables.

Strong correlations can affect the interpretation of linear model coefficients and may indicate redundant environmental information.

In [31]:
# Calculate feature correlation matrix

correlation_matrix = X.corr()

display(
    correlation_matrix.round(2)
)

,rainfall_7d,rainfall_30d,rainfall_90d,temp_mean_7d,dewpoint_mean_7d,wind_mean_7d,temp_same_day,dewpoint_same_day,wind_same_day,ndvi_nearest_composite,ndvi_anomaly,elevation_m,slope_deg,dist_to_water_m
rainfall_7d,1.00,0.73,0.21,-0.36,0.57,0.42,-0.37,0.51,0.34,0.06,0.14,-0.10,0.04,-0.05
rainfall_30d,0.73,1.00,0.39,-0.21,0.47,0.29,-0.25,0.45,0.24,-0.01,0.06,-0.18,0.03,-0.19
rainfall_90d,0.21,0.39,1.00,-0.21,0.48,0.19,-0.26,0.47,0.13,-0.25,-0.24,-0.31,-0.01,-0.20
temp_mean_7d,-0.36,-0.21,-0.21,1.00,-0.48,-0.14,0.88,-0.36,-0.13,-0.29,-0.12,-0.48,-0.46,-0.07
dewpoint_mean_7d,0.57,0.47,0.48,-0.48,1.00,0.51,-0.53,0.91,0.40,-0.09,0.04,-0.39,-0.02,-0.15
wind_mean_7d,0.42,0.29,0.19,-0.14,0.51,1.00,-0.21,0.48,0.66,-0.01,0.21,-0.28,-0.01,-0.10
temp_same_day,-0.37,-0.25,-0.26,0.88,-0.53,-0.21,1.00,-0.56,-0.32,-0.21,-0.08,-0.36,-0.45,-0.07
dewpoint_same_day,0.51,0.45,0.47,-0.36,0.91,0.48,-0.56,1.00,0.51,-0.09,0.05,-0.43,-0.02,-0.13
wind_same_day,0.34,0.24,0.13,-0.13,0.40,0.66,-0.32,0.51,1.00,0.10,0.22,-0.18,0.02,0.05
ndvi_nearest_composite,0.06,-0.01,-0.25,-0.29,-0.09,-0.01,-0.21,-0.09,0.10,1.00,0.80,0.48,0.39,0.23


## 19. Multicollinearity Diagnostic

Calculate Variance Inflation Factor (VIF) to quantify redundancy among environmental predictors.

High VIF values indicate that a feature contains information that overlaps substantially with other predictors, which can make individual Logistic Regression coefficients difficult to interpret.

In [33]:
# Calculate VIF without requiring statsmodels

from sklearn.linear_model import LinearRegression
import numpy as np

vif_results = []

for feature in X.columns:
    other_features = X.drop(columns=[feature])
    target_feature = X[feature]

    model = LinearRegression()
    model.fit(other_features, target_feature)

    r_squared = model.score(other_features, target_feature)

    if r_squared >= 0.999999:
        vif = np.inf
    else:
        vif = 1 / (1 - r_squared)

    vif_results.append({
        "feature": feature,
        "VIF": vif
    })

vif_results = (
    pd.DataFrame(vif_results)
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

display(vif_results.round(2))

,feature,VIF
0,dewpoint_mean_7d,14.45
1,dewpoint_same_day,14.10
2,temp_mean_7d,11.62
3,temp_same_day,10.81
4,elevation_m,7.25
5,ndvi_nearest_composite,4.93
6,ndvi_anomaly,4.21
7,rainfall_7d,3.02
8,rainfall_30d,2.67
9,wind_mean_7d,2.57


## 20. Feature Redundancy Sensitivity Test

Evaluate whether removing highly redundant same-day weather variables materially changes model performance.

This is a diagnostic experiment rather than final feature selection.

In [34]:
# Define a reduced feature set

reduced_features = [
    "rainfall_7d",
    "rainfall_30d",
    "rainfall_90d",
    "temp_mean_7d",
    "dewpoint_mean_7d",
    "wind_mean_7d",
    "ndvi_nearest_composite",
    "ndvi_anomaly",
    "elevation_m",
    "slope_deg",
    "dist_to_water_m"
]

X_reduced = X[reduced_features]

print("Original features:", X.shape[1])
print("Reduced features:", X_reduced.shape[1])
print("\nRemoved:")
print(set(X.columns) - set(reduced_features))

Original features: 14
Reduced features: 11

Removed:
{'dewpoint_same_day', 'wind_same_day', 'temp_same_day'}


## 21. Reduced Logistic Regression Validation

Evaluate Logistic Regression using the reduced 11-feature dataset while preserving the same grouped, stratified five-fold validation strategy.

This tests whether the model's performance depends substantially on the highly redundant same-day weather variables.

In [35]:
# Reduced Logistic Regression — grouped 5-fold validation

reduced_logistic_results = []

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(X_reduced, y, groups=groups),
    start=1
):
    X_train = X_reduced.iloc[train_idx]
    X_val = X_reduced.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    reduced_logistic_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0)
    })

reduced_logistic_results_df = pd.DataFrame(
    reduced_logistic_results
)

display(reduced_logistic_results_df)

print("\nMean metrics:")
print(
    reduced_logistic_results_df[
        [
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1"
        ]
    ].mean().round(3)
)

,fold,accuracy,balanced_accuracy,precision,recall,f1
0,1,0.884615,0.881481,0.838710,0.962963,0.896552
1,2,0.750000,0.750000,0.782609,0.692308,0.734694
2,3,0.846154,0.840000,0.771429,1.000000,0.870968
3,4,0.942308,0.941481,0.928571,0.962963,0.945455
4,5,0.941176,0.942308,1.000000,0.884615,0.938776



Mean metrics:
accuracy             0.873
balanced_accuracy    0.871
precision            0.864
recall               0.901
f1                   0.877
dtype: float64


## 22. Reduced Random Forest Validation

Evaluate Random Forest using the reduced 11-feature dataset and the same grouped, stratified five-fold validation strategy.

The result will be compared with both the original Random Forest and reduced Logistic Regression.

In [36]:
# Reduced Random Forest — grouped 5-fold validation

reduced_rf_results = []

for fold, (train_idx, val_idx) in enumerate(
    sgkf.split(X_reduced, y, groups=groups),
    start=1
):
    X_train = X_reduced.iloc[train_idx]
    X_val = X_reduced.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    reduced_rf_results.append({
        "fold": fold,
        "accuracy": accuracy_score(y_val, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred, zero_division=0),
        "recall": recall_score(y_val, y_pred, zero_division=0),
        "f1": f1_score(y_val, y_pred, zero_division=0)
    })

reduced_rf_results_df = pd.DataFrame(
    reduced_rf_results
)

display(reduced_rf_results_df)

print("\nMean metrics:")
print(
    reduced_rf_results_df[
        [
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall",
            "f1"
        ]
    ].mean().round(3)
)

,fold,accuracy,balanced_accuracy,precision,recall,f1
0,1,0.942308,0.940000,0.900000,1.000000,0.947368
1,2,0.884615,0.884615,0.916667,0.846154,0.880000
2,3,0.711538,0.707407,0.687500,0.814815,0.745763
3,4,0.865385,0.865926,0.884615,0.851852,0.867925
4,5,0.960784,0.961538,1.000000,0.923077,0.960000



Mean metrics:
accuracy             0.873
balanced_accuracy    0.872
precision            0.878
recall               0.887
f1                   0.880
dtype: float64


## 23. Fold-Level Performance Stability

Inspect validation performance across individual folds to determine whether the reduced models perform consistently rather than relying on a small number of unusually strong folds.

In [37]:
# Compare fold-level F1 scores

fold_comparison = pd.DataFrame({
    "Fold": range(1, 6),
    "Logistic Regression F1": (
        reduced_logistic_results_df["f1"].values
    ),
    "Random Forest F1": (
        reduced_rf_results_df["f1"].values
    )
})

display(
    fold_comparison.round(3)
)

print("\nF1 standard deviation:")
print(
    "Logistic Regression:",
    round(
        reduced_logistic_results_df["f1"].std(),
        3
    )
)

print(
    "Random Forest:",
    round(
        reduced_rf_results_df["f1"].std(),
        3
    )
)

,Fold,Logistic Regression F1,Random Forest F1
0,1,0.897,0.947
1,2,0.735,0.880
2,3,0.871,0.746
3,4,0.945,0.868
4,5,0.939,0.960



F1 standard deviation:
Logistic Regression: 0.085
Random Forest: 0.085


## 24. Model Improvement Over Baseline

Quantify how much the selected presentation models improve over the majority-class baseline.

In [38]:
# Compare models against the majority baseline

baseline_balanced_accuracy = 0.500

comparison = pd.DataFrame({
    "Model": [
        "Majority Baseline",
        "Reduced Logistic Regression",
        "Reduced Random Forest"
    ],
    "Balanced Accuracy": [
        baseline_balanced_accuracy,
        reduced_logistic_results_df["balanced_accuracy"].mean(),
        reduced_rf_results_df["balanced_accuracy"].mean()
    ]
})

comparison["Improvement_vs_Baseline"] = (
    comparison["Balanced Accuracy"]
    - baseline_balanced_accuracy
)

comparison["Relative_Improvement_%"] = (
    comparison["Improvement_vs_Baseline"]
    / baseline_balanced_accuracy
    * 100
)

display(comparison.round(3))

,Model,Balanced Accuracy,Improvement_vs_Baseline,Relative_Improvement_%
0,Majority Baseline,0.500,0.000,0.000
1,Reduced Logistic Regression,0.871,0.371,74.211
2,Reduced Random Forest,0.872,0.372,74.379


## 25. Final Ngao Presentation Model Configuration

The presentation model is frozen using the reduced 11-feature representation identified during feature redundancy analysis.

Random Forest is selected as the primary presentation model based on the strongest overall cross-validation metrics, while Logistic Regression remains an interpretable reference model.

These models are for presentation/demo purposes and are not the production QueleaGuard prediction system.

In [39]:
# Freeze the final presentation configuration

FINAL_FEATURES = list(X_reduced.columns)

FINAL_MODEL_NAME = "Random Forest"

print("Final model:", FINAL_MODEL_NAME)
print("Number of features:", len(FINAL_FEATURES))
print("\nFinal features:")

for i, feature in enumerate(FINAL_FEATURES, start=1):
    print(f"{i:2}. {feature}")

Final model: Random Forest
Number of features: 11

Final features:
 1. rainfall_7d
 2. rainfall_30d
 3. rainfall_90d
 4. temp_mean_7d
 5. dewpoint_mean_7d
 6. wind_mean_7d
 7. ndvi_nearest_composite
 8. ndvi_anomaly
 9. elevation_m
10. slope_deg
11. dist_to_water_m


## 26. Train Final Ngao Presentation Model

Train the selected Random Forest using the frozen 11-feature configuration.

The complete dataset is used only after model selection and validation have been completed. The previously reported cross-validation metrics remain the performance estimates for the model.

In [40]:
# Train the final Ngao Presentation Random Forest

from sklearn.ensemble import RandomForestClassifier

X_final = X_reduced[FINAL_FEATURES]
y_final = y.copy()

final_rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

final_rf_model.fit(X_final, y_final)

print("Final model trained successfully.")
print("Training rows:", len(X_final))
print("Features:", X_final.shape[1])
print("Model:", FINAL_MODEL_NAME)

Final model trained successfully.
Training rows: 259
Features: 11
Model: Random Forest


## 27. Save Ngao Presentation Model Artifacts

Save the trained Random Forest, feature configuration, and model metadata so the presentation model can be reproduced and loaded without retraining.

In [41]:
from pathlib import Path

MODEL_DIR = PROJECT_ROOT / "models" / "ngao_presentation"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Model output directory:")
print(MODEL_DIR)

print("\nDirectory exists:", MODEL_DIR.exists())

Model output directory:
c:\Users\pc\Projects\QueleaGuard\models\ngao_presentation

Directory exists: True


## 28. Save Final Random Forest Artifact

Persist the trained Random Forest model so it can be loaded later for the Ngao Presentation demo without retraining.

In [42]:
import joblib

MODEL_PATH = MODEL_DIR / "ngao_presentation_random_forest.joblib"

joblib.dump(final_rf_model, MODEL_PATH)

print("Model saved successfully.")
print("Path:")
print(MODEL_PATH)

print("\nFile exists:", MODEL_PATH.exists())
print("File size:", round(MODEL_PATH.stat().st_size / 1024, 2), "KB")

Model saved successfully.
Path:
c:\Users\pc\Projects\QueleaGuard\models\ngao_presentation\ngao_presentation_random_forest.joblib

File exists: True
File size: 964.81 KB


## 29. Save Ngao Presentation Model Metadata

Store the final model configuration, feature set, validation results, and dataset information alongside the trained artifact.

In [43]:
import json

metadata = {
    "project": "QueleaGuard",
    "model_purpose": "Ngao Presentation / Demo",
    "model_type": "Random Forest",
    "artifact": "ngao_presentation_random_forest.joblib",

    "dataset": {
        "path": "data/processed/ngao_presentation_dataset.csv",
        "rows": int(len(X_final)),
        "features_original": 14,
        "features_final": len(FINAL_FEATURES),
        "presence_rows": int((y_final == 1).sum()),
        "pseudo_absence_rows": int((y_final == 0).sum())
    },

    "features": FINAL_FEATURES,

    "removed_features": [
        "temp_same_day",
        "dewpoint_same_day",
        "wind_same_day"
    ],

    "validation": {
        "method": "StratifiedGroupKFold",
        "n_splits": 5,
        "group_definition": "identical environmental feature profiles",
        "group_leakage": False,
        "conflicting_repeated_profile_labels": False
    },

    "cross_validation_metrics": {
        "accuracy": 0.873,
        "balanced_accuracy": 0.872,
        "precision": 0.878,
        "recall": 0.887,
        "f1": 0.880
    },

    "baseline": {
        "type": "majority_class",
        "balanced_accuracy": 0.500,
        "balanced_accuracy_improvement": 0.372
    },

    "training_configuration": {
        "n_estimators": 300,
        "random_state": 42,
        "n_jobs": -1
    },

    "status": "presentation_model_only",
    "production_model": False
}

METADATA_PATH = MODEL_DIR / "ngao_presentation_model_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved successfully.")
print("Path:")
print(METADATA_PATH)

print("\nFile exists:", METADATA_PATH.exists())

Metadata saved successfully.
Path:
c:\Users\pc\Projects\QueleaGuard\models\ngao_presentation\ngao_presentation_model_metadata.json

File exists: True


## 30. Verify Saved Model Artifact

Reload the persisted Random Forest artifact from disk and verify that it produces valid predictions using the exact frozen feature configuration.

In [44]:
# Reload the saved model from disk

loaded_rf_model = joblib.load(MODEL_PATH)

print("Saved model loaded successfully.")
print("Model type:", type(loaded_rf_model).__name__)
print("Number of trees:", loaded_rf_model.n_estimators)
print("Expected features:", loaded_rf_model.n_features_in_)

Saved model loaded successfully.
Model type: RandomForestClassifier
Number of trees: 300
Expected features: 11


In [45]:
# Prediction smoke test

sample_X = X_final.iloc[:5]

sample_predictions = loaded_rf_model.predict(sample_X)
sample_probabilities = loaded_rf_model.predict_proba(sample_X)[:, 1]

smoke_test = pd.DataFrame({
    "actual": y_final.iloc[:5].values,
    "prediction": sample_predictions,
    "presence_probability": sample_probabilities
})

display(smoke_test.round(3))

print("Prediction test successful.")

,actual,prediction,presence_probability
0,1,1,1.000
1,1,1,1.000
2,1,1,1.000
3,1,1,1.000
4,1,1,0.987


Prediction test successful.


## 32. Validate Final Model Schema

Confirm that the persisted presentation model and metadata agree on the exact 11-feature input schema.

In [46]:
# Validate model and metadata schema

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

metadata_features = loaded_metadata["features"]

print("Model expects:", loaded_rf_model.n_features_in_, "features")
print("Metadata lists:", len(metadata_features), "features")
print("Feature list matches:", metadata_features == FINAL_FEATURES)

print("\nFinal feature schema:")
for i, feature in enumerate(metadata_features, start=1):
    print(f"{i:2}. {feature}")

assert loaded_rf_model.n_features_in_ == len(metadata_features)
assert metadata_features == FINAL_FEATURES

print("\n✓ Model and metadata schema are consistent.")

Model expects: 11 features
Metadata lists: 11 features
Feature list matches: True

Final feature schema:
 1. rainfall_7d
 2. rainfall_30d
 3. rainfall_90d
 4. temp_mean_7d
 5. dewpoint_mean_7d
 6. wind_mean_7d
 7. ndvi_nearest_composite
 8. ndvi_anomaly
 9. elevation_m
10. slope_deg
11. dist_to_water_m

✓ Model and metadata schema are consistent.


## 33. Ngao Presentation Prediction

Use the saved Random Forest artifact to generate a Quelea presence prediction from an environmental profile.

The prediction uses the exact 11-feature schema established during model validation.

In [47]:
# Ngao Presentation prediction function

def predict_quelea_presence(environmental_profile, model=loaded_rf_model):
    """
    Predict Quelea presence for a single environmental profile.

    Parameters
    ----------
    environmental_profile : dict
        Dictionary containing the 11 required environmental features.

    model : sklearn model
        Trained Ngao Presentation Random Forest model.

    Returns
    -------
    dict
        Prediction, probability, and input profile.
    """

    # Check that all required features are present
    missing_features = [
        feature
        for feature in FINAL_FEATURES
        if feature not in environmental_profile
    ]

    if missing_features:
        raise ValueError(
            f"Missing required features: {missing_features}"
        )

    # Build input dataframe in the exact feature order
    input_df = pd.DataFrame(
        [environmental_profile],
        columns=FINAL_FEATURES
    )

    # Generate prediction
    prediction = int(model.predict(input_df)[0])

    # Probability of Quelea presence
    probability = float(
        model.predict_proba(input_df)[0, 1]
    )

    return {
        "prediction": prediction,
        "presence_probability": probability,
        "environmental_profile": input_df
    }


print("Prediction function created successfully.")

Prediction function created successfully.


## 34. Example Environmental Profile

Demonstrate the prediction interface using an environmental profile from the Ngao Presentation dataset.

This example is for demonstrating model operation; validation performance is based exclusively on the grouped cross-validation results reported earlier.

In [48]:
# Select one environmental profile for demonstration

demo_profile = X_final.iloc[0].to_dict()

print("Demo environmental profile:")

for feature, value in demo_profile.items():
    print(f"{feature:25} : {value}")

Demo environmental profile:
rainfall_7d               : 70.49
rainfall_30d              : 310.08
rainfall_90d              : 573.74
temp_mean_7d              : 27.81
dewpoint_mean_7d          : 15.28
wind_mean_7d              : 1.86
ndvi_nearest_composite    : 0.6869
ndvi_anomaly              : 0.1463
elevation_m               : 1158.0
slope_deg                 : 0.98
dist_to_water_m           : 231.0


In [49]:
# Generate Ngao Presentation prediction

demo_result = predict_quelea_presence(demo_profile)

prediction_label = (
    "QUELEA PRESENCE"
    if demo_result["prediction"] == 1
    else "NO QUELEA PRESENCE"
)

probability = demo_result["presence_probability"]

print("=" * 55)
print("QUELEAGUARD — NGAO PRESENTATION PREDICTION")
print("=" * 55)

print(f"\nPrediction: {prediction_label}")
print(f"Presence probability: {probability:.1%}")

QUELEAGUARD — NGAO PRESENTATION PREDICTION

Prediction: QUELEA PRESENCE
Presence probability: 100.0%


## 36. Out-of-Fold Prediction Demonstration

Generate predictions for validation observations using models that did not train on those observations.

This provides a more realistic demonstration of model behavior and avoids presenting a training-row prediction as evidence of generalization.

In [50]:
# Generate out-of-fold predictions using the reduced Random Forest

oof_predictions = np.zeros(len(X_reduced))
oof_probabilities = np.zeros(len(X_reduced))

for train_idx, val_idx in sgkf.split(
    X_reduced,
    y,
    groups=groups
):
    X_train = X_reduced.iloc[train_idx]
    X_val = X_reduced.iloc[val_idx]

    y_train = y.iloc[train_idx]

    fold_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    fold_model.fit(X_train, y_train)

    oof_predictions[val_idx] = fold_model.predict(X_val)
    oof_probabilities[val_idx] = (
        fold_model.predict_proba(X_val)[:, 1]
    )

print("Out-of-fold predictions generated.")
print("Predictions:", len(oof_predictions))
print("Probabilities:", len(oof_probabilities))

Out-of-fold predictions generated.
Predictions: 259
Probabilities: 259


## 37. Out-of-Fold Prediction Results

Combine the environmental profiles, observed labels, out-of-fold predictions, and predicted presence probabilities into a single evaluation table.

These predictions represent model behavior on observations that were excluded from the corresponding training fold.

In [51]:
# Build out-of-fold prediction results

oof_results = X_reduced.copy()

oof_results["actual_presence"] = y.values
oof_results["predicted_presence"] = oof_predictions.astype(int)
oof_results["presence_probability"] = oof_probabilities

display(
    oof_results[
        [
            "actual_presence",
            "predicted_presence",
            "presence_probability"
        ]
    ].head(10).round(3)
)

print("Out-of-fold result rows:", len(oof_results))

,actual_presence,predicted_presence,presence_probability
0,1,1,0.990
1,1,1,0.990
2,1,1,0.990
3,1,1,0.990
4,1,1,0.920
5,1,1,0.997
6,1,1,0.987
7,1,1,0.990
8,1,1,0.997
9,1,1,0.990


Out-of-fold result rows: 259


## 38. Representative Out-of-Fold Predictions

Identify representative correctly classified presence and pseudo-absence observations for presentation demonstration.

Examples are selected from out-of-fold predictions rather than from the final model's training predictions.

In [52]:
# Find correctly classified out-of-fold examples

correct_presence = oof_results[
    (oof_results["actual_presence"] == 1) &
    (oof_results["predicted_presence"] == 1)
].copy()

correct_absence = oof_results[
    (oof_results["actual_presence"] == 0) &
    (oof_results["predicted_presence"] == 0)
].copy()

print("Correct presence predictions:", len(correct_presence))
print("Correct pseudo-absence predictions:", len(correct_absence))

print("\nPresence examples:")
display(
    correct_presence[
        ["actual_presence", "predicted_presence", "presence_probability"]
    ]
    .sort_values("presence_probability")
    .iloc[[len(correct_presence)//2]]
    .round(3)
)

print("\nPseudo-absence examples:")
display(
    correct_absence[
        ["actual_presence", "predicted_presence", "presence_probability"]
    ]
    .sort_values("presence_probability")
    .iloc[[len(correct_absence)//2]]
    .round(3)
)

Correct presence predictions: 118
Correct pseudo-absence predictions: 108

Presence examples:


,actual_presence,predicted_presence,presence_probability
25,1,1,0.947



Pseudo-absence examples:


,actual_presence,predicted_presence,presence_probability
159,0,0,0.087


## 39. Environmental Profiles Behind the Demonstration Predictions

Inspect the environmental conditions associated with the selected out-of-fold presence and pseudo-absence examples.

In [53]:
# Retrieve the environmental profiles for the two presentation examples

presence_example_idx = 25
absence_example_idx = 159

presence_profile = X_reduced.loc[presence_example_idx]
absence_profile = X_reduced.loc[absence_example_idx]

print("=" * 65)
print("OBSERVED PRESENCE EXAMPLE — ROW 25")
print("=" * 65)

for feature, value in presence_profile.items():
    print(f"{feature:25} : {value}")

print("\n" + "=" * 65)
print("PSEUDO-ABSENCE EXAMPLE — ROW 159")
print("=" * 65)

for feature, value in absence_profile.items():
    print(f"{feature:25} : {value}")

OBSERVED PRESENCE EXAMPLE — ROW 25
rainfall_7d               : 18.94
rainfall_30d              : 109.53
rainfall_90d              : 324.89
temp_mean_7d              : 26.72
dewpoint_mean_7d          : 15.23
wind_mean_7d              : 1.82
ndvi_nearest_composite    : 0.5411
ndvi_anomaly              : -0.0313
elevation_m               : 1158.0
slope_deg                 : 0.98
dist_to_water_m           : 231.0

PSEUDO-ABSENCE EXAMPLE — ROW 159
rainfall_7d               : 79.06
rainfall_30d              : 187.97
rainfall_90d              : 381.98
temp_mean_7d              : 26.19
dewpoint_mean_7d          : 17.7
wind_mean_7d              : 3.31
ndvi_nearest_composite    : 0.6576
ndvi_anomaly              : 0.1576
elevation_m               : 1227.0
slope_deg                 : 4.71
dist_to_water_m           : 838.0


## 40. Ngao Presentation Prediction Demo

A reusable prediction interface for demonstrating how environmental conditions are converted into a Quelea presence probability.

In [59]:
def ngao_presentation_predict(profile):
    """
    Generate an estimated Quelea presence probability
    from an 11-feature environmental profile.
    """

    result = predict_quelea_presence(profile)

    probability = result["presence_probability"]
    prediction = result["prediction"]

    return {
        "prediction": (
            "QUELEA PRESENCE"
            if prediction == 1
            else "NO QUELEA PRESENCE"
        ),
        "presence_probability": probability
    }

In [61]:
absence_demo = ngao_presentation_predict(
    absence_profile.to_dict()
)

print("=" * 60)
print("NGAO PRESENTATION — PSEUDO-ABSENCE EXAMPLE")
print("=" * 60)

print(f"Prediction           : {absence_demo['prediction']}")
print(
    f"Presence probability : "
    f"{absence_demo['presence_probability']:.1%}"
)

NGAO PRESENTATION — PSEUDO-ABSENCE EXAMPLE
Prediction           : NO QUELEA PRESENCE
Presence probability : 2.0%


In [58]:
absence_demo = ngao_presentation_predict(
    absence_profile.to_dict()
)

print("=" * 60)
print("NGAO PRESENTATION — PSEUDO-ABSENCE EXAMPLE")
print("=" * 60)

print(f"Prediction           : {absence_demo['prediction']}")
print(
    f"Presence probability : "
    f"{absence_demo['presence_probability']:.1%}"
)
print(f"Risk category        : {absence_demo['risk_category']}")

NGAO PRESENTATION — PSEUDO-ABSENCE EXAMPLE
Prediction           : NO QUELEA PRESENCE
Presence probability : 2.0%
Risk category        : VERY LOW
